# Getting started

In [1]:
# Use the os package to interact with the environment
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import glob
import sys as sys

In [2]:
# show all columns when printing
pd.set_option('display.max_columns', None) 

In [3]:
#set releases
RAW_GENO = '/home/jupyter/workspace/path/to/release7'
CLINICAL = '/home/jupyter/workspace/path/to/release7/clinical_data'
RELATED = '/home/jupyter/workspace/path/to/release7/meta_data/related_samples'

# Install Packages

In [4]:
%%capture
%%bash

# Install plink 2.0
cd /home/jupyter/
if test -e /home/jupyter/plink2; then

echo "Plink2 is already installed in /home/jupyter/"
else
echo "Plink2 is not installed"
cd /home/jupyter/

wget https://s3.amazonaws.com/plink2-assets/alpha6/plink2_linux_x86_64_20250122.zip

unzip -o plink2_linux_x86_64_20250122.zip

fi

In [5]:
%%bash

# chmod plink 2 to make sure you have permission to run the program
chmod u+x /home/jupyter/plink2

# Assocication analysis of LRRK2 p.L1795F (chr12:40322386:G:T) with PD in European population

In [6]:
# prepare covariant
key = pd.read_csv(f'{CLINICAL}/master_key_release7_final_vwb.csv')
print(key.shape)

(58446, 26)


In [7]:
# Subsetting to keep only a few columns
key = key[['GP2sampleID', 'baseline_GP2_phenotype_for_qc', 'biological_sex_for_qc', 'age_at_sample_collection', 'age_of_onset','family_history_for_qc','label']]
# Renaming the columns
key.rename(columns = {'GP2sampleID':'IID',
                      'baseline_GP2_phenotype_for_qc':'phenotype',
                      'age_at_sample_collection':'AGE', 
                      'age_of_onset':'AAO'}, inplace = True)

# subset EUR
eur_key=key.loc[key['label']=='EUR']


In [8]:
eur_key['family_history_for_qc'].value_counts()

family_history_for_qc
Not Reported    21812
No              14204
Yes              5277
Unknown           141
Name: count, dtype: int64

In [9]:
# Load information about related individuals in the EUR samples
related_df = pd.read_csv(f'{RELATED}/EUR_release7_vwb.related')

# Make a list of just one set of related people
related_list = list(related_df['IID1'])

# Check value counts of related and remove only one related individual
eur_key = eur_key[~eur_key["IID"].isin(related_list)]

# Convert phenotype to binary (Control==1/PD==2)
eur_key['PHENO'] = np.where(eur_key['phenotype'] == 'PD', 2, 
                            np.where(eur_key['phenotype'] == 'Control', 1, np.nan))

eur_key=eur_key.loc[~eur_key['PHENO'].isnull()]

# Convert sex to binary (male ==1/ female ==2)
eur_key['SEX'] = np.where(eur_key['biological_sex_for_qc'] == 'Female', 2, 
                            np.where(eur_key['biological_sex_for_qc'] == 'Male', 1, np.nan))

# Create AGE_covariate for association
eur_key['AGE_covariate'] = eur_key['AAO'].fillna(eur_key['AGE'])

# Recode family_history_for_qc 
eur_key['family_history'] = np.where(eur_key['family_history_for_qc'] == 'Yes', 2, 
                            np.where(eur_key['family_history_for_qc'] == 'No', 1, np.nan))

## Get the PCs
pcs = pd.read_csv(f'{RAW_GENO}/EUR/EUR_release7_vwb.eigenvec', sep='\t')

#Select just first 5 PCs
selected_columns = ['IID', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5','PC6']
pcs = pd.DataFrame(data=pcs.iloc[:, 1:8].values, columns=selected_columns)

# Drop the first row (since it's now the column names)
pcs = pcs.drop(0)

# Reset the index to remove any potential issues
pcs = pcs.reset_index(drop=True)

## Make covariate file
df = pd.merge(eur_key, pcs, on='IID', how='left')

#Make additional columns - FID, fatid and matid
df['FID'] = 0
df['FATID'] = 0
df['MATID'] = 0

## Clean up and keep columns we need
final_df = df[['FID','IID', 'FATID', 'MATID', 'SEX', 'AGE_covariate','family_history','PHENO','PC1', 'PC2', 'PC3', 'PC4', 'PC5','PC6']].copy()
print(f'Number of total samples: {final_df.shape[0]}')
final_df.to_csv('eur_covariate_files.txt',sep='\t',na_rep='NA', index=False)

#Check number of PD cases missing age
pd_missAge = final_df[(final_df['PHENO']==2)&(final_df['AGE_covariate'].isna())]
print(f'Number of PD cases missing age: {pd_missAge.shape[0]}')

#Check number of controls missing age
control_missAge = final_df[(final_df['PHENO']==1)&(final_df['AGE_covariate'].isna())]
print(f'Number of controls missing age: {control_missAge.shape[0]}')

## Make file of sample IDs to keep
samples_toKeep = final_df[['FID', 'IID']].copy()
samples_toKeep.to_csv('eur_samplestoKeep.txt', sep = '\t', index=False, header=None)

# write out phenotype
final_df[['FID', 'IID','PHENO']].to_csv('eur_pheno_association.txt', sep = '\t', index=False)

Number of total samples: 31113
Number of PD cases missing age: 3974
Number of controls missing age: 3347


In [ ]:
%%bash

#NBA

RAW_GENO='/home/jupyter/workspace/path/to/release7/raw_genotypes'


/home/jupyter/plink2 --pfile ${RAW_GENO}/EUR/EUR_release7_vwb \
                     --keep eur_samplestoKeep.txt \
                     --pheno eur_pheno_association.txt \
                     --glm \
                     --ci 0.95 \
                     --snp Seq_rs111910483.2_ilmnrev_ilmnF2BT \
                     --covar eur_covariate_files.txt \
                     --covar-name SEX,AGE_covariate,family_history,PC1,PC2,PC3,PC4,PC5,PC6 \
                     --covar-variance-standardize \
                     --out eur_lrrk2_l1795f


In [11]:
# read association

df=pd.read_csv('eur_lrrk2_l1795f.PHENO.glm.logistic.hybrid',sep='\t')
df

,#CHROM,POS,ID,REF,ALT,PROVISIONAL_REF?,A1,OMITTED,A1_FREQ,FIRTH?,TEST,OBS_CT,OR,LOG(OR)_SE,L95,U95,Z_STAT,P,ERRCODE
0,12,40322386,Seq_rs111910483.2_ilmnrev_ilmnF2BT,C,A,Y,A,C,0.000156,Y,ADD,16058,0.840293,1.489990,0.045305,15.585300,-0.116782,9.070330e-01,.
1,12,40322386,Seq_rs111910483.2_ilmnrev_ilmnF2BT,C,A,Y,A,C,0.000156,Y,SEX,16058,0.736277,0.019624,0.708495,0.765148,-15.600700,7.204000e-55,.
2,12,40322386,Seq_rs111910483.2_ilmnrev_ilmnF2BT,C,A,Y,A,C,0.000156,Y,AGE_covariate,16058,0.524573,0.023726,0.500738,0.549542,-27.193100,7.837260e-163,.
3,12,40322386,Seq_rs111910483.2_ilmnrev_ilmnF2BT,C,A,Y,A,C,0.000156,Y,family_history,16058,1.536110,0.023750,1.466250,1.609310,18.074200,5.090940e-73,.
4,12,40322386,Seq_rs111910483.2_ilmnrev_ilmnF2BT,C,A,Y,A,C,0.000156,Y,PC1,16058,0.967983,0.023741,0.923973,1.014090,-1.370640,1.704870e-01,.
5,12,40322386,Seq_rs111910483.2_ilmnrev_ilmnF2BT,C,A,Y,A,C,0.000156,Y,PC2,16058,0.795213,0.026910,0.754358,0.838281,-8.515190,1.663240e-17,.
6,12,40322386,Seq_rs111910483.2_ilmnrev_ilmnF2BT,C,A,Y,A,C,0.000156,Y,PC3,16058,0.838745,0.030282,0.790412,0.890033,-5.807000,6.360280e-09,.
7,12,40322386,Seq_rs111910483.2_ilmnrev_ilmnF2BT,C,A,Y,A,C,0.000156,Y,PC4,16058,1.158850,0.025009,1.103420,1.217070,5.895020,3.746260e-09,.
8,12,40322386,Seq_rs111910483.2_ilmnrev_ilmnF2BT,C,A,Y,A,C,0.000156,Y,PC5,16058,1.132800,0.024387,1.079930,1.188260,5.112940,3.171770e-07,.
9,12,40322386,Seq_rs111910483.2_ilmnrev_ilmnF2BT,C,A,Y,A,C,0.000156,Y,PC6,16058,1.010510,0.024483,0.963161,1.060180,0.426862,6.694800e-01,.
